# Chromatic PSF from a field star

Fit a point source in wavelength bands, describe its chromatic shape with smooth trends, and save a compact reusable model. The supplied `data/fieldstar.fits` contains a real MUSE field-star cutout, including flux and variance.

In [ ]:
%matplotlib inline

from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from astropy.modeling.models import Planar2D
from astropy.convolution import discretize_model
from astropy.visualization import ImageNormalize, AsinhStretch
from IPython.display import display

from prism.data import Cube
from prism.psf import fitimage, fitbands, fittrends, ChromaticPSF
from prism.modeling.models import EllipticalMoffatPSF

plt.rcParams['axes.xmargin'] = 0

In [ ]:
cubepath = Path('data/fieldstar.fits')
outpath = Path('results/psf')
outpath.mkdir(parents=True, exist_ok=True)

cube = Cube.from_fits(cubepath, ext_values='DATA', ext_var='VAR', dtype=float)
{"shape": cube.shape, "wavelength_range": cube.z[[0, -1]]*cube.zunit}

# Apply any channel masks here: cube.mask uses True for valid samples.

In [ ]:
img = cube.get_image(method='mean')
ny, nx = img.shape
y, x = np.indices(img.shape)
cy, cx = np.unravel_index(np.nanargmax(img.values), img.shape)

fig, ax = plt.subplots(figsize=(5, 4))
norm = ImageNormalize(vmin=np.nanpercentile(img.values, 1),
                      vmax=np.nanpercentile(img.values, 99.8), stretch=AsinhStretch())
im = ax.imshow(img.values, origin='lower', norm=norm, cmap='viridis')
ax.scatter(cx, cy, color='red', marker='x', label='Initial center')
ax.set(xlabel='x [pixel]', ylabel='y [pixel]', title='Field star — mean image')
ax.legend()
fig.colorbar(im, ax=ax, label=f'Flux [{cube.unit}]')
plt.show()

## Choose a spatial profile

Use `MoffatPSF` for a circular profile or `EllipticalMoffatPSF` for two principal-axis widths and a rotation angle. Both use FWHM and `beta`, which controls the wings; `amplitude` is the peak brightness and `theta` is in radians. A separate `Planar2D` describes the local background. Only the source profile contributes to the normalized PSF.

These models follow the Astropy modeling interface: set parameter values, `fixed` flags and `bounds` explicitly. Native Astropy `Gaussian2D` and `Moffat2D` profiles are also supported. Choose a profile that captures the observed shape without adding parameters the data cannot constrain; orientation is poorly determined for a nearly circular source.

Initialize near the image peak, then fit the mean image with `fitimage` to obtain starting values for the band fits. `init=None` keeps the supplied guesses; optional `init='peak'` seeds free position and amplitude parameters from the selected image region. `factor` controls integration of the model over each pixel; the observed cube is not resampled.

In [ ]:
model = EllipticalMoffatPSF(
    amplitude=float(img.values[cy, cx] - np.nanmedian(img.values)),
    x_0=float(cx), y_0=float(cy), fwhm_x=3.4, fwhm_y=3, beta=2.5, theta=0,
)
model.x_0.bounds = (cx - 3, cx + 3)
model.y_0.bounds = (cy - 3, cy + 3)
model.fwhm_x.bounds = (0.3, 12)
model.fwhm_y.bounds = (0.3, 12)
model.theta.bounds = (-np.pi/2, np.pi/2)
model.beta.bounds = (1.05, 12)
background = Planar2D(intercept=float(np.nanmedian(img.values)))
factor = 5

Inspect the mean-image fit's convergence and residual statistic before using it to initialize the independent band fits.

In [ ]:
initial = fitimage(img, model, background=background, factor=factor)
initial

In [ ]:
model.parameters = np.array([initial[name] for name in model.param_names])
background.parameters = np.array([initial[name] for name in background.param_names])

In [ ]:
bands = fitbands(cube, model, bands=20, background=background, factor=factor)
bands.meta['source'] = str(cubepath)

## Describe the wavelength dependence

Fit the PSF independently in wavelength bands, then describe each measured parameter with a smooth trend. Choose the flexibility for each parameter in `trends`:

| `basis` | Dictionary values | Practical choice |
| --- | --- | --- |
| `'bspline'` (default) | Number of coefficients | Piecewise smooth curves with local flexibility. Useful for curved trends; coefficient bounds keep predictions within the specified physical limits. |
| `'polynomial'` | Polynomial degree | A simple global trend. Useful for low-order behavior; higher degrees can produce strong curvature near the edges. |

For example, a spline with 5 coefficients and a polynomial of degree 4 each fit five coefficients. Start with modest flexibility and inspect the trend plots; neither choice guarantees that a more flexible fit is more accurate.

Review the band measurements before adopting the model. Select table rows explicitly to exclude unsuitable bands; there is no automatic rejection. The model covers the wavelength range of the accepted bands, with less constraint near the outer edges.

Save the compact model with `psf.save`. It can then generate normalized PSF images at any wavelength within `psf.domain`, without reloading or refitting the star cube. A finite image contains only the fraction of the PSF falling inside it; this is the quantity needed for an aperture correction.

The cubic B-spline repeats each endpoint knot four times; five coefficients give one interior knot. This bounds values through the coefficients but does not impose zero slope or curvature. Edge channels lie beyond the first/last band centers and are less constrained: inspect them and reduce flexibility if needed.

In [ ]:
trends = {'fwhm_x': 5, 'fwhm_y': 5, 'beta': 5, 'x_0': 5, 'y_0': 5, 'theta': 5}
basis = 'bspline'  # Counts coefficients; 5 matches the budget of polynomial degree 4.
psf = fittrends(bands, trends, basis=basis, display=True, save=outpath / 'fitted_parameters.pdf')
psf.save(outpath / 'psf.fits', overwrite=True)

# Results of the spatial fits in each band (success is their convergence flag).
print(f"Spatial band fits: {np.count_nonzero(psf.bands['success'])}/{len(psf.bands)} converged")
display(psf.bands[['nband', 'wave', 'success', 'fwhm_x', 'fwhm_y', 'theta', 'beta', 'x_0', 'y_0', 'redchi2']])

## Resolution versus wavelength

`psf.fwhm(wave)` returns the intrinsic principal-axis widths in `(x, y)` order, in pixels. The following conversion uses the square-pixel scale retained in the demonstration FITS header (`PIXSCALE`, arcseconds per pixel). Rotation does not change these intrinsic widths. This represents a resolution curve.

In [ ]:
wave = np.linspace(*psf.domain, 200) * psf.unit
scale = cube.header['PRIMARY']['PIXSCALE'] * u.arcsec
widths = psf.fwhm(wave).to_value(u.pix) * scale

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(wave.to_value(u.AA), widths[:, 0].value, label='Principal x')
ax.plot(wave.to_value(u.AA), widths[:, 1].value, label='Principal y')
ax.set(xlabel='Wavelength [Å]', ylabel='FWHM [arcsec]')
ax.legend()
fig.tight_layout()
fig.savefig(outpath / 'resolution.pdf')
plt.show()

## Inspect one band

Change `band` below. This reconstructs the independent spatial fit, including the background, not the smoothed trend prediction. Data and model share a display scale. The absolute residual divided by the stellar peak is a useful first check of reconstruction quality; it does not by itself bound aperture-correction errors.

In [ ]:
band = 8
display(bands[band])

img = cube.get_image(weights=bands['channels'][band], method='mean')
fitted = model.copy()
for name in model.param_names:
    getattr(fitted, name).value = bands[name][band]
fitted_background = background.copy()
for name in background.param_names:
    getattr(fitted_background, name).value = bands[name][band]

# Astropy midpoint integration, matching factor used by fitbands.
fitted_image = discretize_model(fitted, x_range=(0, nx), y_range=(0, ny),
                               mode='oversample', factor=factor)
fitted_image += fitted_background(x, y)
residuals = np.where(img.mask, img.values - fitted_image, np.nan)

In [ ]:
norm = ImageNormalize(vmin=np.nanpercentile(img.values, 1),
                      vmax=np.nanpercentile(img.values, 99.8), stretch=AsinhStretch())
limit = np.nanpercentile(np.abs(residuals), 99)
fig, axes = plt.subplots(1, 3, figsize=(13, 4), constrained_layout=True)
for ax, image, title in zip(axes[:2], [img.values, fitted_image], ['Data', 'PSF + plane']):
    im = ax.imshow(image, origin='lower', norm=norm, cmap='viridis')
    ax.set_title(title)
    fig.colorbar(im, ax=ax, label=f'Flux [{cube.unit}]')
im = axes[2].imshow(residuals, origin='lower', vmin=-limit, vmax=limit, cmap='RdBu_r')
axes[2].set_title('Data − model')
fig.colorbar(im, ax=axes[2], label=f'Residual [{cube.unit}]')
for ax in axes:
    ax.set(xlabel='x [pixel]', ylabel='y [pixel]')
fig.suptitle(f"Band {band}: {bands['lower'][band]:.0f}–{bands['upper'][band]:.0f} {cube.zunit}")
fig.savefig(outpath / 'band-check.png', dpi=150)
plt.show()

In [ ]:
print(f'Peak absolute residual / fitted stellar peak: '
      f'{np.nanmax(np.abs(residuals)) / np.max(fitted_image-fitted_background(x,y)):.3%}')

## Use the PSF on another source

Reload the saved model with `ChromaticPSF.from_fits`, generate an image at the target position, or evaluate the FWHM at selected wavelengths. The target must share the calibration's spatial sampling and orientation, and wavelengths must lie within `psf.domain`.

Continue with [Aperture Correction](Aperture%20Correction.ipynb) to extract a spectrum and compare aperture correction with PSF-plus-background fitting.

In [ ]:
saved = ChromaticPSF.from_fits(outpath / "psf.fits")
image = saved.get_image(6500 * u.AA, (26, 26), center=(13, 12))
saved.fwhm([5000, 6500, 8000] * u.AA)